# Lagrange coefficients $f, g$ (and $\dot f, \dot g$)

**Goal:** package the universal anomaly $\chi$ and the Stumpff functions into the four numbers that advance a state vector in time, via $\mathbf{r}=f\,\mathbf{r}_0+g\,\mathbf{v}_0$ and $\mathbf{v}=\dot f\,\mathbf{r}_0+\dot g\,\mathbf{v}_0$.

**Code (answer key):** [`f_and_g`, `fDot_and_gDot`](../curtis/algorithms/lagrange_coefficients.py) · **Book:** §3.7, Appendix D.6 · Equations 3.66

Like the Stumpff functions, this is a building block (no single worked-example number). We verify it by the invariant $f\dot g-\dot f g=1$ and by checking it reproduces the propagation of Example 3.7.

## Read first

| Symbol | Meaning |
|---|---|
| $f, g$ | Lagrange coefficients: $\mathbf{r}=f\,\mathbf{r}_0+g\,\mathbf{v}_0$ |
| $\dot f, \dot g$ | their time derivatives: $\mathbf{v}=\dot f\,\mathbf{r}_0+\dot g\,\mathbf{v}_0$ |
| $\chi$ | universal anomaly (from `kepler_U`, 3.3) |
| $z=\alpha\chi^2$ | Stumpff argument; $\alpha=1/a$ |
| $r_0,\ r$ | radius at the start and at time $t$ |

**The four coefficients (Eqs 3.66):**
$$f=1-\frac{\chi^2}{r_0}C(z),\quad g=\Delta t-\frac{\chi^3}{\sqrt{\mu}}S(z),\quad \dot f=\frac{\sqrt{\mu}}{r\,r_0}\big(zS(z)-1\big)\chi,\quad \dot g=1-\frac{\chi^2}{r}C(z).$$

## The picture (why only four numbers)

Angular momentum $\mathbf{h}=\mathbf{r}_0\times\mathbf{v}_0$ is constant, so the orbit never leaves the plane spanned by $\mathbf{r}_0$ and $\mathbf{v}_0$. That means the position at *any* later time is some combination
$$\mathbf{r}(t)=f(t)\,\mathbf{r}_0+g(t)\,\mathbf{v}_0,$$
and differentiating gives $\mathbf{v}(t)=\dot f\,\mathbf{r}_0+\dot g\,\mathbf{v}_0$. So the entire time evolution of a 6-component state collapses to **four scalar functions of time** — that is the Lagrange-coefficient idea.

They are not independent: the two-body flow preserves phase-space area, which forces
$$\boxed{\,f\dot g-\dot f g=1\,}\quad\text{for all }t.$$
This single identity is both a deep conservation law and the cleanest correctness test you have.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, "..")
# Prerequisites you already built: the Stumpff functions and the universal
# Kepler solver. Imported here so this notebook focuses on f, g themselves.
from curtis.algorithms.stumpff import stumpC, stumpS
from curtis.algorithms.alg_3_3_kepler_U import kepler_U

In [ ]:
# Watch f, gdot vary over one orbit while the combination f*gdot - fdot*g stays 1.
# (Example 3.7's orbit: an ellipse.)
mu = 398600.0
r0v = np.array([7000.0, -12124.0, 0.0])
v0v = np.array([2.6679, 4.6210, 0.0])
r0  = np.linalg.norm(r0v)
vr0 = np.dot(r0v, v0v)/r0
alpha = 2/r0 - np.linalg.norm(v0v)**2/mu      # > 0  -> ellipse
T = 2*np.pi/np.sqrt(mu) * (1/alpha)**1.5       # period

ts = np.linspace(0, T, 300)
f_, gd_, W_ = [], [], []
for t in ts:
    x = kepler_U(t, r0, vr0, alpha, mu)
    z = alpha*x**2; C = stumpC(z); S = stumpS(z)
    f = 1 - x**2/r0*C
    g = t - x**3/np.sqrt(mu)*S
    R = f*r0v + g*v0v; r = np.linalg.norm(R)
    fdot = np.sqrt(mu)/(r*r0)*(z*S - 1)*x
    gdot = 1 - x**2/r*C
    f_.append(f); gd_.append(gdot); W_.append(f*gdot - fdot*g)

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.axhline(1, color='#c2c0b6', lw=0.9)
ax.plot(ts/T, f_,  color='#378ADD', lw=2, label='f')
ax.plot(ts/T, gd_, color='#D85A30', lw=2, label='ġ')
ax.plot(ts/T, W_,  color='#1D9E75', lw=2.5, label='f·ġ − ḟ·g')
ax.set_xlabel('time  (fraction of one orbit)')
ax.set_title('The coefficients vary, but f·ġ − ḟ·g stays exactly 1')
ax.legend(loc='center left')
plt.show()

## See it — the invariant is dead flat

$f$ (blue) and $\dot g$ (orange) swing through a full range over the orbit — the state really is being mixed and rotated. But the green curve, $f\dot g-\dot f g$, sits pinned at **1** the whole way around. That flat line is conservation of phase-space area (the map is *symplectic*), and it is what guarantees the propagation is exactly reversible: knowing the four coefficients lets you step forward *or* backward with no loss.

## Where these equations come from

### The form $\mathbf{r}=f\mathbf{r}_0+g\mathbf{v}_0$
Because $\mathbf{r}_0$ and $\mathbf{v}_0$ span the (fixed) orbital plane, any in-plane vector — in particular $\mathbf{r}(t)$ — is a linear combination of them. Writing $\mathbf{r}=f\mathbf{r}_0+g\mathbf{v}_0$ and differentiating gives $\mathbf{v}=\dot f\mathbf{r}_0+\dot g\mathbf{v}_0$. Initial conditions fix $f(0)=1,\ g(0)=0,\ \dot f(0)=0,\ \dot g(0)=1$.

### The closed forms (Eqs 3.66)
Solving the two-body problem in the universal variable $\chi$ — substituting $r=r_0 f+\dots$ into $\ddot{\mathbf r}=-\mu\mathbf r/r^3$ and integrating — yields the four coefficients in terms of $\chi$ and the Stumpff functions:
$$f=1-\frac{\chi^2}{r_0}C(z),\quad g=\Delta t-\frac{\chi^3}{\sqrt{\mu}}S(z),\quad \dot f=\frac{\sqrt{\mu}}{r\,r_0}\big(zS-1\big)\chi,\quad \dot g=1-\frac{\chi^2}{r}C(z).$$
The same $C(z), S(z)$ from before reappear — that is the universal formulation paying off again.

### Why $f\dot g-\dot f g=1$
This Wronskian is the determinant of the linear map $(\mathbf r_0,\mathbf v_0)\mapsto(\mathbf r,\mathbf v)$ restricted to the plane. Two-body motion is Hamiltonian, so the map preserves area: the determinant is $1$ for all $t$. Equivalently, it is conservation of $\mathbf h$ in disguise.

## Step by step (in code order)

**`f_and_g(x, t, ro, alpha, mu)`** — with $z=\alpha x^2$:
$$f=1-\frac{x^2}{r_0}C(z),\qquad g=t-\frac{x^3}{\sqrt{\mu}}S(z).$$

**`fDot_and_gDot(x, r, ro, alpha, mu)`** — with $z=\alpha x^2$ (note this one needs the *current* radius $r$):
$$\dot f=\frac{\sqrt{\mu}}{r\,r_0}\big(zS(z)-1\big)x,\qquad \dot g=1-\frac{x^2}{r}C(z).$$

**↓ Now type both functions below.** Straight formula transcription — no loops. Mind that `f_and_g` uses the starting radius `ro`, while `fDot_and_gDot` also needs the current radius `r`. Answer key linked above; peek only after you try.

In [ ]:
def f_and_g(x, t, ro, alpha, mu):
    """Lagrange coefficients f, g  (Eqs 3.66a-b).  x = universal anomaly, alpha = 1/a."""
    z = alpha*x**2
    # f = 1 - x**2/ro * stumpC(z)            (Eq 3.66a)
    # g = t - x**3/sqrt(mu) * stumpS(z)      (Eq 3.66b)
    # return f, g
    raise NotImplementedError("fill in f and g, then delete this line")


def fDot_and_gDot(x, r, ro, alpha, mu):
    """Time derivatives fdot, gdot  (Eqs 3.66c-d).  r = current radius."""
    z = alpha*x**2
    # fdot = sqrt(mu)/(r*ro) * (z*stumpS(z) - 1) * x   (Eq 3.66c)
    # gdot = 1 - x**2/r * stumpC(z)                    (Eq 3.66d)
    # return fdot, gdot
    raise NotImplementedError("fill in fdot and gdot, then delete this line")

## Verify — the invariant, and Example 3.7

We feed in the orbit and elapsed time of Example 3.7, get $\chi$ from `kepler_U`, then check two things: the Lagrange map's determinant is $1$, and it actually propagates $(\mathbf r_0,\mathbf v_0)$ to the published Example 3.7 state.

**Example 3.7 final state:** $\mathbf r=[-3297.77,\ 7413.40,\ 0]$ km, $\;\mathbf v=[-8.2976,\ -0.964045,\ 0]$ km/s.

Run once both functions are typed.

In [ ]:
mu  = 398600.0
r0v = np.array([7000.0, -12124.0, 0.0])
v0v = np.array([2.6679, 4.6210, 0.0])
t   = 3600.0

r0  = np.linalg.norm(r0v)
vr0 = np.dot(r0v, v0v)/r0
alpha = 2/r0 - np.linalg.norm(v0v)**2/mu
x   = kepler_U(t, r0, vr0, alpha, mu)

f, g = f_and_g(x, t, r0, alpha, mu)
R = f*r0v + g*v0v
r = np.linalg.norm(R)
fdot, gdot = fDot_and_gDot(x, r, r0, alpha, mu)
V = fdot*r0v + gdot*v0v

print(f"f, g       = {f:.6g}, {g:.6g}")
print(f"fdot, gdot = {fdot:.6g}, {gdot:.6g}")
print(f"Wronskian f*gdot - fdot*g = {f*gdot - fdot*g:.12g}   (expect 1)")
print(f"r (km)   = [{R[0]:.6g} {R[1]:.6g} {R[2]:.6g}]   (expect [-3297.77  7413.40  0])")
print(f"v (km/s) = [{V[0]:.6g} {V[1]:.6g} {V[2]:.6g}]   (expect [-8.2976  -0.964045  0])")

assert abs(f*gdot - fdot*g - 1) < 1e-10                       # the invariant
assert np.allclose(R, [-3297.77, 7413.40, 0.0], atol=1e-2)    # matches Example 3.7
assert np.allclose(V, [-8.2976, -0.964045, 0.0], atol=1e-4)
print("\nAll checks passed ✔")

## What this confirms

- A two-body state evolves as a **linear map** of $(\mathbf r_0,\mathbf v_0)$ — four numbers, because the motion is planar.
- $f\dot g-\dot f g=1$ holds exactly (you just used it as a test): the propagator is area-preserving and reversible.
- The same $\chi$, $C(z)$, $S(z)$ machinery feeds straight in — this is the last piece of the propagation engine.

**Next:** Algorithm 3.4 (`rv_from_r0v0`) — the capstone. It wires together `kepler_U` (get $\chi$ from $\Delta t$) and these coefficients (turn $\chi$ into the new state) to propagate any orbit forward in time. You just previewed its exact result.